# 13 - Applied Text Analysis

In this notebook, we will applied some of the techniques that we have seen in the previous sessions. We start by building a corpus with texts extracted from the Project Gutenberg. We chose medical periodicals. We want to check if the gender analysis performed on novels is applicable when we work with non-fiction using computational analysis.

As always we start with our usual importations. We first define our module_path, grant access to our drive and import the questions and solutions for this notebook.

In [ ]:
# @title Grant GoogleColab access to your GoogleDrive and import questions for this notebook
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add your module folder to Python path
import sys
module_path = f"/content/drive/My Drive/IDH/Notebooks"
sys.path.append(module_path)
print("GoogleColab can now access your GoogleDrive.")

# Import exercises
from QuestionsPGmed import E1, E2, E3, Q1, Q2, question, solution

We then import the necessary libraries.

In [ ]:
import time
import csv
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from collections import Counter

### Exercise 1

Create a Python Pandas dataframe called `pg_med` with the CSV file called `pg_medperiodicals.csv` that you will find the folder `PG`. Follow the way we created dataframes in the previous notebooks, notably notebooks 10 and 11. You will pass as argument `index_col=0` to keep the index column of the original file. The argument must come after the path to the file with a comma.

In [ ]:
pg_med = pd.read_csv(f"{module_path}/PG/pg_medperiodicals.csv", index_col=0)


In [ ]:
solution(E1)

In [ ]:
# Print the first raws of the dataframe to see the content
pg_med.head(5)

We will use BeautifulSoup and the ID in the CSV file that we have just stored in our dataframe to scrape the texts from the Project Gutenberg. We create a folder in which we will store our files.

In [ ]:
OUT_DIR = Path(f"{module_path}/PG/pg_medperiodicals")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Define headers
headers = {
        "User-Agent": "Introduction to Digital Humanities 2025 (contact: mbednarkiewicz@faculty.ie.edu)",
        "Accept-Language": "en",
    }

We define a function to fetch the text matching the given ID.

In [ ]:
def fetch_text_from_id(pgid: int, timeout=20) -> str:
    """
    Returns the text matching the input id.
    """
    url = f"https://www.gutenberg.org/cache/epub/{pgid}/pg{pgid}.txt"
    resp = requests.get(url, headers=headers, timeout=timeout)
    return resp.text

### Exercise 2

Analyse the function `fetch_text_from_id` and answer the questions.

In [ ]:
question(Q1)

In [ ]:
question(Q2)

### Exercise 3

Fetch the text with IS 22336 using the function fetch_text_from_id. Keep the `timeout` as it is.

In [ ]:
# Assign the output of fetch_text_from_id to the given variable
text_22336 =

In [ ]:
# Test your result
print(text_22336[:1500])

If you did not get the first 1500 words printed, check the solution below and run the cells again.

In [ ]:
solution(E2)

Let's try our function and store the texts in the folder OUT_DIR that we have created. We test with two texts first.

In [ ]:
# Trial with two ids
ids = [22336, 25819]

In [ ]:
for pgid in ids:
    text = fetch_text_from_id(int(pgid))
    out_path = OUT_DIR / f"pg{pgid}.txt"
    out_path.write_text(text, encoding="utf-8")
    time.sleep(1)  # be polite to PG

Open now the folder `PG/pg_medperiodicals` and check if the two texts are present. If so, you can run the two following cells.

In [ ]:
# Get all ids from our matadata dataframe in a list
ids = pd.Index(pg_med.index).astype(int)

In [ ]:
# Loop over the IDs and scrape the corresponging texts
for pgid in ids:
    text = fetch_text_from_id(int(pgid))
    out_path = OUT_DIR / f"pg{pgid}.txt"
    out_path.write_text(text, encoding="utf-8")
    print(f"pg{pgid}.txt has been downloaded")
    time.sleep(1)  # be polite to PG

Now that we have our texts stored in the folder called `pg_medperiodicals`, we can perform some computational analysis on the files content. We want to count pronouns and see what are the most common words. For that we first remove the punctuation. We define a folder in which we will store the cleaned texts.

In [ ]:
IN_DIR = Path(f"{module_path}/PG/pg_medperiodicals")
OUT_DIR = Path(f"{module_path}/PG/pg_medperiodicals_clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~’‘“”'

def clean_text(text: str) -> str:
    text_clean = text  # start from the original
    for mark in punctuation:
        text_clean = text_clean.replace(mark, '')
    return text_clean

Python strings have a method `.translate()` that applies a translation table created by `str.maketrans`. If we map every punctuation mark to `None` or an empty string `''`, they’ll be removed in one go. The method `translate` is used the same way as the method `replace`.

Advantages:

- Much faster on big texts (no Python loop, all handled in C).

- Code is cleaner and straightforward.

In [ ]:
# Build the translation table:
# each chararacter in punctuation is matched to None
table = str.maketrans('', '', punctuation)

In [ ]:
# We create a smaple text to test the method
sample_test = "Dr. John’s Medicine: Ancient & Modern!"

In [ ]:
sample_test.translate(table)

### Exercise 4

Write a function called `trans_text` that will take as argument `text` which is of type `str` (i.e. string) and will return the `text` _translated_ with the `table` that we have just defined.

In [ ]:
# Write and run your function


In [ ]:
# Test your function
print(trans_text(sample_test))

In [ ]:
solution(E3)

We can now _clean_ our corpus and remove all punctuation signs from the texts using a `loop`.

In [ ]:
for file in IN_DIR.glob("*.txt"):
    raw = file.read_text(encoding="utf-8", errors="ignore")
    cleaned = trans_text(raw)
    out_path = OUT_DIR / file.name   # same filename, new folder
    out_path.write_text(cleaned, encoding="utf-8")
    print(f"Cleaned {file.name} → {out_path}")

When we work with large number of files, we must always check that what we think we did actually happened. Let's check that the punctuation was indeed removed in one random text from our new folder: `pg_medperiodicals_clean`.

In [ ]:
# pick the first file in the input folder
fname = next(IN_DIR.glob("*.txt")).name

In [ ]:
# We will compare the raw and the clean texts
raw = (IN_DIR / fname).read_text(encoding="utf-8", errors="ignore")
clean = (OUT_DIR / fname).read_text(encoding="utf-8", errors="ignore")

In [ ]:
# build a 2-column DataFrame with just the first 300 chars
df_check = pd.DataFrame({
    "RAW": [raw[:300]],
    "CLEAN": [clean[:300]]
})

In [ ]:
from IPython.display import display

print("Comparing file:", fname)
display(df_check.style.hide(axis="index"))

Contrary to what we did in notebook 8, we kept here the capital letters and did not apply the Python string method `.lower()` that we applied in notebook 8 to get all words in lower case. If your analysis invovled proper names, you might want to keep the capital letters, which will be an easy way to recognise a person in your text. Imagine you want to investigate which scientific authorities are mentioned in the medical corpus. In this case, you will use the capital letters. But if you want to count the most frequent tokens, the capital letters might create obstacles. Let's illustrate this problem. We will count the tokens in our corpus of medical periodicals and display those with the highest count. We will count them keeping the capital letters and then removing them, that is lowercasing the whole text to observe the difference.

#### Without lower-case

In [ ]:
token_counter = Counter()

for file in OUT_DIR.glob("*.txt"):
    text = file.read_text(encoding="utf-8", errors="ignore")
    tokens = text.split()   # naive tokenization on whitespace
    token_counter.update(tokens)

print("Total unique tokens:", len(token_counter), "\n")
for word, count in token_counter.most_common(20):
    print(f"{word:15} {count}")

#### With lower-case

In [ ]:
token_lower_counter = Counter()

for file in OUT_DIR.glob("*.txt"):
    text = file.read_text(encoding="utf-8", errors="ignore")
    tokens = text.lower().split()   # lower-case before tokenizing & naïve tokenisation on white space
    token_lower_counter.update(tokens)

In [ ]:
print("Total unique tokens:", len(token_lower_counter), "\n")
for word, count in token_lower_counter.most_common(20):
    print(f"{word:15} {count}")

As you can see, there are 34847 tokens when the text is not lower-case and 29649 when all the words are lower-case. This means that 5198 words are duplicate and are counts twice when the text is **not** lower-case, once with a capital letter and once without, like **The** and **the** in the list we printed above. It is important to take such details in consideration to produce accurate statistics. Our corpus does **not** have 34847 unique tokens, but 29649.

### Text analysis

We want now to analyse the usage of pronouns in our corpus of medical periodicals. We are interested in exploring the frequence of masculine pronouns against feminine ones for example. But we might also find relevant to investigate how the pronouns for the first person singular and plural, **I** and **we**, are used, to show how subjectivity and objectivity are treated in science.

We first create two lists of stopwords that we want to analyse, one that includes pronouns in order to see what are the most prominent words in our corpus outside the pronouns, and one that excludes the pronouns so that we can count them.

In [ ]:
# --- Stopword lists ---
stopwords_with_pronouns = [
    'i','me','my','myself','we','our','ours','ourselves','you','youre','youve','youll','youd',
    'your','yours','yourself','yourselves', 'he', 'his', 'himself', 'she', 'her', 'hers', 'herself',
    'it','its','itself','they','them','their','theirs', 'themselves',
    'what','which','who','whom','this','that','thatll','these','those',
    'am','is','are','was','were','be','been','being','have','has','had','having',
    'do','does','did','doing','a','an','the','and','but','if','or','because','as','until','while',
    'of','at','by','for','with','about','against','between','into','through','during','before',
    'after','above','below','to','from','up','down','in','out','on','off','over','under',
    'again','further','then','once','here','there','when','where','why','how','all','any',
    'both','each','few','more','most','other','some','such','no','nor','not','only','own',
    'same','so','than','too','very','s','t','can','will','just','don','dont','should',
    'shouldve','now','d','ll','m','o','re','ve','y','ain','aren','arent','couldn','couldnt',
    'didn','didnt','doesn','doesnt','hadn','hadnt','hasn','hasnt','haven','havent','isn',
    'isnt','ma','mightn','mightnt','mustn','mustnt','needn','neednt','shan','shant',
    'shouldn','shouldnt','wasn','wasnt','weren','werent','won','wont','wouldn','wouldnt',
    # extras
    'said','thou','one','went','came','thee','could','would','took','go','shall','must','however','thy'
]

stopwords_no_pronouns = [
    'what','which','who','whom','this','that','thatll','these','those',
    'am','is','are','was','were','be','been','being','have','has','had','having',
    'do','does','did','doing','a','an','the','and','but','if','or','because','as','until','while',
    'of','at','by','for','with','about','against','between','into','through','during','before',
    'after','above','below','to','from','up','down','in','out','on','off','over','under',
    'again','further','then','once','here','there','when','where','why','how','all','any',
    'both','each','few','more','most','other','some','such','no','nor','not','only','own',
    'same','so','than','too','very','s','t','can','will','just','don','dont','should',
    'shouldve','now','d','ll','m','o','re','ve','y','ain','aren','arent','couldn','couldnt',
    'didn','didnt','doesn','doesnt','hadn','hadnt','hasn','hasnt','haven','havent','isn',
    'isnt','ma','mightn','mightnt','mustn','mustnt','needn','neednt','shan','shant',
    'shouldn','shouldnt','wasn','wasnt','weren','werent','won','wont','wouldn','wouldnt',
    # extras
    'said','one','went','came','could','would','took','go','shall','must','however','thy'
]

We create now a function to count tokens that are **not** in our stopwords list.

In [ ]:
def count_tokens(stopwords: list):
  """
  Count the tokens that are not in the stopwords list.
  """
  counter = Counter()
  files = list(OUT_DIR.glob("*.txt"))
  for file in files:
      text = file.read_text(encoding="utf-8", errors="ignore")
      tokens = text.lower().split()
      tokens = [tok for tok in tokens if tok not in stopwords]
      counter.update(tokens)
  return counter

We count the tokens with and without the pronouns using our function and store the top 20 tokens in a dataframe.

In [ ]:
counts_with = count_tokens(stopwords_with_pronouns)
counts_no = count_tokens(stopwords_no_pronouns)

top = 20

if counts_with and counts_no:
    top_with = counts_with.most_common(top)
    top_no = counts_no.most_common(top)

    df_compare = pd.DataFrame({
        "Without pronouns": [f"{w} ({c})" for w,c in top_with],
        "With pronouns": [f"{w} ({c})" for w,c in top_no]
    })

display(df_compare)

### Visualisation

We can now visualise our data to get a quick glimpse.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# --- Top N ---
topN = 20
df = pd.DataFrame(counts_with.most_common(topN), columns=["token", "count"])

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(df["token"], df["count"])
plt.title(f"Top {topN} words (stopwords removed, pronouns removed)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# --- Pronoun groups (you can adapt/expand) ---
pronoun_groups = {
    "I (1-sg)": {"i", "me", "my", "mine", "myself"},
    "We (1-pl)": {"we", "us", "our", "ours", "ourselves"},
    "You (2)": {"you", "your", "yours", "yourself", "yourselves"},
    "He (3-sg-m)": {"he", "him", "his", "himself"},
    "She (3-sg-f)": {"she", "her", "hers", "herself"},
    "It (3-sg-n)": {"it", "its", "itself"},
    "They (3-pl)": {"they", "them", "their", "theirs", "themselves"},
}

# --- Aggregate counts per group ---
group_counts = {group: sum(counts_no[p] for p in forms)
                for group, forms in pronoun_groups.items()}

# --- Make DataFrame ---
df_pronouns = pd.DataFrame(list(group_counts.items()), columns=["Pronoun group", "Count"]).sort_values("Count", ascending=False)

# --- Plot ---
plt.figure(figsize=(10,6))
plt.bar(df_pronouns["Pronoun group"], df_pronouns["Count"])
plt.title("Pronoun usage (groups agglomerated)")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

df_pronouns


---
## Lesson Summary

- We write functions when we need to parse a large quantity of data, like texts.
- We create an extra forlder to separate original raw texts from their clean versions.
- When removing punctuations or stopwords from a corpus, we must think carefully of the consequences.
- Visualisations in DataFrames or graphs are useful to perform preliminary text analysis and test hypotheses.